In [41]:
import sys
from pathlib import Path

print(sys.executable)
racine_projet = Path.cwd().parent
sys.path.append(str(racine_projet))

/home/onyxia/work/.venv/bin/python


# 1. Clé de jointure

Clé de jointure : code_uic

L'unicité de la clé dans la table gares-de-voyageurs.parquet a été vérifiée dans la fonction *clean_gares()* (étape 5)

On vérifie le nombre de code_uic par ligne dans la table *gares-de-voyageurs*

In [42]:
from src.nettoyage import clean_frequentation
from src.nettoyage import clean_gares
from src.verification_jointures import verif_code_uic

base = [clean_gares(),clean_frequentation()]

verif_code_uic(base)


       position_geographique code_commune  code_uic
0      49.6852237, 1.7743058        60001  87313759
1     47.5546432, -1.5244159        44001  87481614
2          50.10221, 1.82449        80001  87317362
3        48.725468, 2.419151        94001  87545269
4      48.9551835, 2.0919031        78551  87386052
...                      ...          ...       ...
2787  48.4702462, -2.6524629        22389  87473116
2788  43.9781851, -0.7361528        40333  87671487
2789   44.9106891, 2.3644474        15267  87645101
2790     49.622035, 0.750115        76758  87413385
2791   47.6943143, 7.2945376        68384  87182196

[2792 rows x 3 columns]
11
                    nom_gare  code_uic code_postal segmentation_drg  annee  \
0                   Abbaretz  87481614       44170                C   2023   
1      Achères Grand Cormier  87386052       78100                B   2023   
2            Achiet-le-Grand  87342048       62121                C   2023   
3                       Agay  877575

Dans la base sgares-de-voyageurs*, 11 gares ont plusieurs *code_uic*. Ces codes multiples n'apparaissent pas dans la base *frequentation-gares* ce qui a posé un problème lors des premières tentatives de jointures (correction à l'étape 2 de la fonction *join_sncf())*

# 2. Type de jointure

Dans un premier temps, nous réalisé la jointure avec comme l'arugment "outer" pour la fonction merge *join_gares_audit()* afin de conserver toutes les gares et de comprendre pourquoi certaines n'apparaissent dans une base ou dans l'autre.

In [43]:
# On récupère le fichier fusionné en appelant la fonction join_sncf_audit depuis src.jointure
from src.jointure import join_sncf_audit

sncf_audit = join_sncf_audit()

# On affiche les dimensions du fichier, les premières lignes et les types des variables
print(sncf_audit.shape)
display(sncf_audit.info())
display(sncf_audit.head())

(21092, 9)
<class 'pandas.DataFrame'>
RangeIndex: 21092 entries, 0 to 21091
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   position_geographique  19503 non-null  str     
 1   code_commune           19503 non-null  str     
 2   code_uic               21092 non-null  str     
 3   nom_gare               21070 non-null  str     
 4   code_postal            21070 non-null  str     
 5   segmentation_drg       20601 non-null  str     
 6   annee                  21070 non-null  float64 
 7   frequentation          21070 non-null  float64 
 8   _merge                 21092 non-null  category
dtypes: category(1), float64(2), str(6)
memory usage: 2.3 MB


None

,position_geographique,code_commune,code_uic,nom_gare,code_postal,segmentation_drg,annee,frequentation,_merge
0,"49.003652, 2.570892",93073,87001479,NaN,NaN,NaN,NaN,NaN,left_only
1,"47.3388167, 0.6528244",37122,87009696,La Douzillère,37300,C,2023.0,6573.0,both
2,"47.3388167, 0.6528244",37122,87009696,La Douzillère,37300,C,2022.0,9903.0,both
3,"47.3388167, 0.6528244",37122,87009696,La Douzillère,37300,C,2021.0,9147.0,both
4,"47.3388167, 0.6528244",37122,87009696,La Douzillère,37300,C,2020.0,7664.0,both


In [44]:
# Lignes présentes uniquement dans gares
gares_non_appariees = sncf_audit[sncf_audit["_merge"] == "left_only"]

display(gares_non_appariees)
print(len(gares_non_appariees))

,position_geographique,code_commune,code_uic,nom_gare,code_postal,segmentation_drg,annee,frequentation,_merge
0,"49.003652, 2.570892",93073,87001479,NaN,NaN,NaN,NaN,NaN,left_only
3557,"48.880185, 2.355151",75110,87271023,NaN,NaN,NaN,NaN,NaN,left_only
3558,"48.880185, 2.355151",75110,87271031,NaN,NaN,NaN,NaN,NaN,left_only
7143,"48.8807466, 2.0790676",78367,87382812,NaN,NaN,NaN,NaN,NaN,left_only
7186,"48.897377, 2.224397",92050,87386011,NaN,NaN,NaN,NaN,NaN,left_only
7334,"48.841172, 2.320514",75115,87391011,NaN,NaN,NaN,NaN,NaN,left_only
7335,"48.841172, 2.320514",75115,87391102,NaN,NaN,NaN,NaN,NaN,left_only
7728,"48.8415189, 2.0618388",78455,87393876,NaN,NaN,NaN,NaN,NaN,left_only
10809,"48.98032, 2.27176",95219,87534131,NaN,NaN,NaN,NaN,NaN,left_only
11223,"48.842285, 2.364891",75113,87547026,NaN,NaN,NaN,NaN,NaN,left_only


22


Au total 22 gares de *gares-de-voyageurs* n'apparaissent pas dans *frequentation-gares*, il n'y a donc pas d'observation de leur fréquentation, il semble donc pertinent de ne pas les garder dans notre base finale.

In [45]:
# Lignes présentes uniquement dans frequentation
frequentation_non_appariees = sncf_audit[sncf_audit["_merge"] == "right_only"]

display(frequentation_non_appariees)
print(len(frequentation_non_appariees))

,position_geographique,code_commune,code_uic,nom_gare,code_postal,segmentation_drg,annee,frequentation,_merge
701,NaN,NaN,87141531,Bainville-sur-Madon,54550,C,2023.0,7.0,right_only
702,NaN,NaN,87141531,Bainville-sur-Madon,54550,C,2022.0,84.0,right_only
703,NaN,NaN,87141531,Bainville-sur-Madon,54550,C,2021.0,6.0,right_only
704,NaN,NaN,87141531,Bainville-sur-Madon,54550,C,2020.0,2.0,right_only
705,NaN,NaN,87141531,Bainville-sur-Madon,54550,C,2019.0,60.0,right_only
...,...,...,...,...,...,...,...,...,...
21065,NaN,NaN,87784892,Bena Fanes,66760,C,2021.0,46.0,right_only
21066,NaN,NaN,87784892,Bena Fanes,66760,C,2020.0,23.0,right_only
21067,NaN,NaN,87784892,Bena Fanes,66760,C,2019.0,78.0,right_only
21068,NaN,NaN,87784892,Bena Fanes,66760,C,2018.0,33.0,right_only


1589


Nous constatons que certaines gares qui n'apparaissent pas dans dans *gares-de-voyageurs* ont fermées (ex : Bainvillle-sur-Madon) mais d'autres semblent encore ouvertes et accueillaient des voyageurs en 2025 (ex : Bena Fanes)

In [46]:
# Nombre de voyageurs dans les lignes non appariées
nb = frequentation_non_appariees["frequentation"].sum()
print(nb)

# Proportion des voyageurs dans les lignes non appariées sur le nombre total de voyageur
tot = clean_frequentation()["frequentation"].sum()
pct = (nb / tot) * 100
print(pct)

52335600.0
0.3071602450599038


Le nombre total de voyageurs passant dans les gares non appariées sur la période étudiée ne représente que 0.3% du nombre total de voyageurs sur cette période. Nous avons donc fait le choix de les négliger et d'utiliser l'argument "inner" dans notre fonction finale *join_sncf()*.

# 3. Résultat

In [47]:
# On récupère le fichier fusionné en appelant la fonction join_sncf depuis src.jointure
from src.jointure import join_sncf

sncf = join_sncf()

# On affiche les premières lignes du fichier
display(sncf.head())

,position_geographique,code_commune,code_uic,nom_gare,code_postal,segmentation_drg,annee,frequentation
0,"49.6852237, 1.7743058",60001,87313759,Abancourt,60220,C,2023,77974
1,"49.6852237, 1.7743058",60001,87313759,Abancourt,60220,C,2022,71517
2,"49.6852237, 1.7743058",60001,87313759,Abancourt,60220,C,2021,51811
3,"49.6852237, 1.7743058",60001,87313759,Abancourt,60220,C,2020,32396
4,"49.6852237, 1.7743058",60001,87313759,Abancourt,60220,C,2019,42685


# 4. Valeurs nulles

